# EAGLE-3 draft for granite-docling

The Medusa route works but loses: `medusa` is in neither `EagleModelTypes` nor the V2-runner
allowlist, so vLLM disables async scheduling *and* falls back to the V1 model runner. Measured, a
drafted round costs 3.33 ms against plain vLLM's 1.20 ms, and break-even lands at 2.78 tokens/step
where the head delivers 1.20 pooled.

`eagle3` keeps both. Measured here with a random-init drafter (acceptance 0, so tok/s *is* the
round cost):

| config | round | drafter adds | break-even |
|---|---:|---:|---:|
| plain vLLM | 1.352 ms | — | 1.000 |
| eagle3 k=1 | 2.147 ms | 0.795 ms | **1.588** |
| eagle3 k=2 | 2.333 ms | 0.981 ms | **1.726** |
| eagle3 k=3 | 2.442 ms | 1.090 ms | 1.807 |
| eagle3 k=5 | 2.970 ms | 1.618 ms | 2.197 |

So eagle3 roughly halves Medusa's bar (2.78 → 1.59 at k=1) but it is **not** the ~1.1 that a
free-drafter calculation suggests: the drafter has a ~0.68 ms fixed cost plus ~0.11 ms per extra
step, and that is paid on every round. Low k is favoured — the marginal token is cheap but the
first one is not.

**What this model is.** Not the latent draft ported over. EAGLE-3 is autoregressive: propose a
token, embed it, feed it back through the drafter's own KV cache, repeat. `Eagle3LlamaForCausalLM`
returns one hidden state per position, so the `[B, T, HORIZON, 576]` parallel-head contract does
not survive — `future_heads` has nowhere to go. What *does* survive is the window: the drafter
attends over its own KV cache, so history comes back without being passed in.

**Prerequisite: traces with EAGLE-3 taps.** The V2 runner sets `use_aux_hidden_state_outputs=True`
unconditionally for `method="eagle3"`, so the target emits layers 2/14/27 concatenated and the
drafter's `fc` maps 1728 → 576. The current corpus stores only `last_hidden_state`, so it must be
re-extracted (4x larger on disk):

```
uv run fastdocling-extract data/images data/traces_taps --keep-taps
```


In [1]:
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch.nn import functional as F
from torchinfo import summary
from tqdm.auto import tqdm

from fastdocling.data import FIRST_OFFSET, corpus_key, iterate_batches, output_vocab, plan, scan_traces, split
from fastdocling.eagle3 import AUX_LAYERS, Eagle3Draft, export_eagle3_checkpoint
from fastdocling.target import MODEL_ID, load_lm_head

# Paths resolve against the repo root, not the working directory: this notebook lives in
# notebooks/eagle3/, so a bare Path("data/traces") would point at notebooks/eagle3/data/traces.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
TRACES = ROOT / "data/traces_taps"   # MUST be a --keep-taps extraction; see the note above
CACHE = ROOT / "data/cache"
OUTPUT = ROOT / "checkpoints/eagle3_draft.pt"
DECODES = CACHE / "decodes"

CONTEXT_LENGTH = 64     # training window; at inference the drafter's own KV cache carries history
BATCH_SIZE = 32
EPOCHS = 1
SPEC_TOKENS = 2         # num_speculative_tokens; k=1..2 is where break-even is lowest
MUON_LR, ADAMW_LR, WARMUP_STEPS = 0.02, 3e-4, 10
VLLM_GPU_FRACTION = 0.35
PRUNE_VOCAB = True      # d2t pruning; unlike Medusa's token_map this is not broken upstream

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
infos = scan_traces(TRACES, min_completion=CONTEXT_LENGTH + 2)
missing = [i for i in infos if not i.has_taps]
if missing:
    raise RuntimeError(
        f"{len(missing)}/{len(infos)} traces in {TRACES} have no EAGLE-3 taps.  vLLM's V2 runner "
        f"always requests aux hidden states for method='eagle3', so the drafter's fc needs "
        f"layers {AUX_LAYERS} concatenated (1728-d).  Re-extract with:\n"
        f"  uv run fastdocling-extract data/images {TRACES.name} --keep-taps")

train_infos, holdout_infos = split(infos, holdout_fraction=0.02)
schedule = plan(train_infos, CONTEXT_LENGTH, BATCH_SIZE, horizon=2)
TOTAL_STEPS = schedule.steps_for_epochs(EPOCHS)
CORPUS_KEY = corpus_key(train_infos)
print(schedule.describe(steps=TOTAL_STEPS))
print(f"holdout: {len(holdout_infos)} pages; corpus key {CORPUS_KEY}")

vocab = output_vocab(train_infos, cache=TRACES / f"output_vocab_{CORPUS_KEY}.npy") if PRUNE_VOCAB else None
if vocab is not None:
    print(f"draft vocabulary pruned to {len(vocab):,} of 100,352 tokens (written as d2t)")

FileNotFoundError: no trace directory at /root/Code/fastdocling/data/traces_taps (cwd /root/Code/fastdocling/notebooks/eagle3); paths are relative to the working directory, so a notebook run from a subdirectory needs an absolute root

## Alignment

At cursor `i` the target's state `h_i` is known, and so is token `i+1` — it is the bonus token the
target already emitted. The drafter embeds that token, combines it with `h_i`, and must predict
token `i+2`. So the drafter's *input* token is at offset 1 and its *label* is at offset 2, which is
`FIRST_OFFSET`, the same convention the latent draft and the live loop use.

`iterate_batches(..., horizon=2, first_offset=1)` yields exactly those two columns:
`tokens[:, :, 0]` is the input embedding's token, `tokens[:, :, 1]` is the label.

In [ ]:
draft = Eagle3Draft(
    in_dim=576 * len(AUX_LAYERS),
    draft_vocab_size=len(vocab) if vocab is not None else 100_352,
).to(device)

# The target's embedding and vocabulary projection are frozen and shared, so they are not trained
# here and are omitted from the export -- vLLM binds the target's own tensors instead.  What is
# left is fc + one decoder layer + norm: the part that actually has to be learned.
trainable = [p for n, p in draft.named_parameters() if not n.startswith(("embed_tokens", "lm_head"))]
for n, p in draft.named_parameters():
    if n.startswith(("embed_tokens", "lm_head")):
        p.requires_grad_(False)
print(f"{sum(p.numel() for p in trainable):,} trainable of {sum(p.numel() for p in draft.parameters()):,} total")

summary(draft, input_data=(torch.zeros(2, CONTEXT_LENGTH, 576 * len(AUX_LAYERS), device=device),
                           torch.zeros(2, CONTEXT_LENGTH, dtype=torch.long, device=device)),
        depth=3, col_names=("input_size", "output_size", "num_params"), row_settings=("var_names",))

In [ ]:
muon_params = [p for p in trainable if p.ndim == 2]
adamw_params = [p for p in trainable if p.ndim != 2]
optimizers = [torch.optim.Muon(muon_params, lr=MUON_LR, weight_decay=0.01, momentum=0.95, nesterov=True)]
if adamw_params:
    optimizers.append(torch.optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=0.01))
warmup_cosine = lambda s: min(1.0, (s + 1) / WARMUP_STEPS) * 0.5 * (1 + np.cos(np.pi * min(1.0, s / max(1, TOTAL_STEPS))))
lr_schedules = [torch.optim.lr_scheduler.LambdaLR(o, warmup_cosine) for o in optimizers]

# Map target token ids onto draft rows when the head is pruned; ids outside the set are ignored.
if vocab is not None:
    target_to_draft = torch.full((100_352,), -100, dtype=torch.long, device=device)
    target_to_draft[torch.as_tensor(vocab, device=device)] = torch.arange(len(vocab), device=device)
else:
    target_to_draft = None


def batches(infos, seed=0, drop_last=True):
    """(aux_states [B,T,1728], input token [B,T], label [B,T]) -- see the alignment note."""
    for x, _, tok in iterate_batches(infos, CONTEXT_LENGTH, BATCH_SIZE, horizon=2, features="eagle3",
                                     seed=seed, device=device, drop_last=drop_last, first_offset=1):
        yield x, tok[:, :, 0], tok[:, :, 1]


@torch.inference_mode()
def acceptance(infos):
    """Teacher-forced top-1 accuracy of the drafter's next-token prediction."""
    draft.eval(); hits = total = 0
    for x, inp, label in batches(infos, seed=0, drop_last=False):
        pred = draft(x, inp).argmax(-1)
        gold = target_to_draft[label] if target_to_draft is not None else label
        mask = gold >= 0
        hits += (pred[mask] == gold[mask]).sum().item(); total += int(mask.sum())
    draft.train()
    return hits / max(1, total)


history, step, started = [], 0, perf_counter()
progress = tqdm(total=TOTAL_STEPS, unit="step", dynamic_ncols=True)
for epoch in range(EPOCHS):
    for x, inp, label in batches(train_infos, seed=epoch):
        gold = target_to_draft[label] if target_to_draft is not None else label
        loss = F.cross_entropy(draft(x, inp).flatten(0, 1), gold.flatten(), ignore_index=-100)
        for o in optimizers:
            o.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        for o, s in zip(optimizers, lr_schedules):
            o.step(); s.step()
        step += 1; progress.update()
        if step % 200 == 0:
            history.append((step, loss.item()))
            progress.set_postfix(loss=f"{loss.item():.3f}", lr=f"{lr_schedules[0].get_last_lr()[0]:.1e}")
progress.close()

p1 = acceptance(holdout_infos)
print(f"\nholdout next-token accuracy {p1:.3f}  ->  k=1 gives {1 + p1:.3f} tokens/step "
      f"(break-even 1.588), k=2 at best {1 + p1 + p1 ** 2:.3f} (break-even 1.726)")
OUTPUT.parent.mkdir(exist_ok=True)
torch.save({"state_dict": draft.state_dict(), "accuracy": p1, "aux_layers": list(AUX_LAYERS)}, OUTPUT)

## Measure it in vLLM

Export writes vLLM's on-disk EAGLE-3 layout (`layers.0.*`, `fc`, `norm`, optional `d2t`) and omits
`embed_tokens`/`lm_head` so the engine binds the target's own. Report **pooled** aggregates — total
tokens over total seconds, total tokens over total rounds. Per-page means were distorted by a
factor of two on this page set, because one short page with high acceptance dominates an unweighted
average.

In [ ]:
from fastdocling.decode import available
from fastdocling.decode.vllm_decoder import sweep_speculative

if "vllm" not in available():
    print("vLLM is not installed here (`uv sync --extra cuda`); skipping.")
else:
    ckpt_dir = export_eagle3_checkpoint(
        draft, CACHE / "eagle3_vllm", vocab=vocab, share_embeddings=vocab is None,
        aux_layers=AUX_LAYERS)
    print(f"exported {ckpt_dir}")

    image_for = lambda info: ROOT.joinpath("data/images", *info.path.stem.split("__")).with_suffix(".png")
    pages = [image_for(i) for i in holdout_infos[:3]]
    cfg = {"method": "eagle3", "model": str(ckpt_dir), "num_speculative_tokens": SPEC_TOKENS}
    rows = sweep_speculative(pages, [None, cfg], cache_dir=DECODES, batched=False,
                             gpu_memory_utilization=VLLM_GPU_FRACTION)

    grouped = {}
    for r in rows:
        grouped.setdefault(r["config"], []).append(r)
    print(f"\n{'configuration':22s}{'tok/s (pooled)':>16s}{'tokens/step':>13s}{'vs plain':>10s}")
    plain = None
    for label, rs in grouped.items():
        tok = sum(r["tokens"] for r in rs)
        sec = sum(r["tokens"] / r["decode_tps"] for r in rs)
        rounds = sum(r["tokens"] / r["tokens_per_round"] for r in rs)
        tps = tok / sec
        plain = plain or tps
        print(f"{label:22s}{tps:16.1f}{tok / rounds:13.3f}{tps / plain:9.2f}x")